# PDF Splitter by Size — Google Colab notebook

**File:** `PDF_Splitter_v1.0_2026-09-08.ipynb`
**Created by Ram Lakhan Meena, INSPECTOR CBIC — 2026-09-08**

## Version log (newest first)

| Version | Date | Changes |
|---|---|---|
| v1.0 | 2026-09-08 | First build. Upload one PDF -> Step 1 measures the real output size of every single page; Step 2 recommends page ranges for a target size (same size for all parts / different size per part / fixed number of parts); Step 3 lets you type your own page ranges manually and re-check the true size of each part; Step 4 splits and downloads (zip or individual files). All sizes shown are **measured**, not guessed — each candidate part is actually written out in memory before its size is reported. |

## How to use (run the cells in order)

1. **Cell 1 — Setup** : installs `pypdf`, loads the engine.
2. **Cell 2 — Upload & analyse** : upload the PDF, see total size, page count and per-page sizes.
3. **Cell 3 — Recommended split** : enter the size you want, get suggested page ranges.
4. **Cell 4 — Manual page selection** : edit the ranges freely (parts can be of different sizes).
5. **Cell 5 — Split & download** : writes the parts and downloads them.

Cell 4 can be used on its own — you do not have to run Cell 3 if you already know the page ranges you want.

**Range syntax used in Cell 4:** `1-12, 13-30, 31-end` (1-based page numbers, both ends included). A single page is just `7`. Overlapping or missing pages are allowed — the notebook only warns you about them.

In [ ]:
# =====================================================================
# CELL 1 - SETUP  (run once per session)
# =====================================================================
import subprocess, sys

try:
    import pypdf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=False)

import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------------
# Engine: all size measurement and split planning lives here.
# ---------------------------------------------------------------------
import io, os, re, zipfile

from pypdf import PdfReader, PdfWriter

MB = 1024 * 1024


def human(nbytes):
    """Readable file size."""
    n = float(nbytes)
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return "%.2f %s" % (n, unit)
        n /= 1024
    return "%.2f TB" % n


class SplitEngine:
    """Measures real output sizes and plans page ranges for a size-based split."""

    def __init__(self, data, filename):
        self.data = data
        self.filename = filename
        self.stem = os.path.splitext(os.path.basename(filename))[0]
        self.reader = PdfReader(io.BytesIO(data))
        self.encrypted = getattr(self.reader, "is_encrypted", False)
        self.n = len(self.reader.pages)
        self.file_size = len(data)
        self.page_sizes = []          # size of each page written out on its own
        self.overhead = 0.0           # bytes double-counted per extra page
        self._cache = {}              # (start, end) -> actual written size
        self.rewrite_size = None      # size of the whole file re-written by pypdf

    # ---------- measurement ----------
    def _write_bytes(self, start, end):
        """Bytes of a PDF containing pages [start, end) (0-based)."""
        writer = PdfWriter()
        for i in range(start, end):
            writer.add_page(self.reader.pages[i])
        buf = io.BytesIO()
        writer.write(buf)
        return buf.getvalue()

    def actual(self, start, end):
        """Measured output size of pages [start, end) in bytes (cached)."""
        key = (start, end)
        if key not in self._cache:
            self._cache[key] = len(self._write_bytes(start, end))
        return self._cache[key]

    def analyse(self, progress=None):
        """Measure every page individually, then calibrate the estimator."""
        self.page_sizes = []
        for i in range(self.n):
            self.page_sizes.append(self.actual(i, i + 1))
            if progress:
                progress(i + 1, self.n)
        self.rewrite_size = self.actual(0, self.n)
        if self.n > 1:
            self.overhead = max(
                0.0, (sum(self.page_sizes) - self.rewrite_size) / float(self.n - 1)
            )
        else:
            self.overhead = 0.0
        return self

    def estimate(self, start, end):
        """Fast estimate of output size for pages [start, end)."""
        k = end - start
        if k <= 0:
            return 0
        return max(0, sum(self.page_sizes[start:end]) - (k - 1) * self.overhead)

    # ---------- planning ----------
    def plan_by_size(self, targets_bytes, max_parts=500):
        """Greedy pack pages into parts. targets_bytes = list of byte limits.

        The list is used part by part; the last value repeats if more parts
        are needed. Returns list of dicts: part, start, end, size, target, ok.
        """
        if not targets_bytes:
            raise ValueError("No target size given.")
        parts, cur, idx = [], 0, 0
        while cur < self.n and len(parts) < max_parts:
            target = targets_bytes[min(idx, len(targets_bytes) - 1)]
            end = cur + 1
            # 1. jump ahead with the cheap estimate
            while end < self.n and self.estimate(cur, end + 1) <= target:
                end += 1
            # 2. shrink until the real measured size fits
            guard = 0
            while end > cur + 1 and self.actual(cur, end) > target and guard < 60:
                end -= 1
                guard += 1
            # 3. grow again if there is still room
            guard = 0
            while end < self.n and self.actual(cur, end + 1) <= target and guard < 60:
                end += 1
                guard += 1
            size = self.actual(cur, end)
            parts.append(
                {
                    "part": len(parts) + 1,
                    "start": cur,
                    "end": end,
                    "size": size,
                    "target": target,
                    "ok": size <= target,
                }
            )
            cur = end
            idx += 1
        return parts

    def plan_by_count(self, n_parts):
        """Split into n_parts with (almost) equal page counts."""
        n_parts = max(1, min(int(n_parts), self.n))
        base, extra = divmod(self.n, n_parts)
        parts, cur = [], 0
        for i in range(n_parts):
            k = base + (1 if i < extra else 0)
            end = cur + k
            parts.append(
                {
                    "part": i + 1,
                    "start": cur,
                    "end": end,
                    "size": self.actual(cur, end),
                    "target": None,
                    "ok": True,
                }
            )
            cur = end
        return parts

    def size_parts(self, ranges):
        """ranges = list of (start, end) 0-based half-open -> measured parts."""
        out = []
        for i, (s, e) in enumerate(ranges):
            out.append(
                {
                    "part": i + 1,
                    "start": s,
                    "end": e,
                    "size": self.actual(s, e),
                    "target": None,
                    "ok": True,
                }
            )
        return out

    # ---------- text <-> ranges ----------
    @staticmethod
    def parts_to_text(parts):
        return ", ".join("%d-%d" % (p["start"] + 1, p["end"]) for p in parts)

    def parse_ranges(self, text):
        """'1-12, 13-30, 45' -> [(0,12),(12,30),(44,45)]. 1-based, inclusive."""
        ranges, errors = [], []
        chunks = [c.strip() for c in re.split(r"[,;\n]+", text) if c.strip()]
        for c in chunks:
            m = re.match(r"^(\d+)\s*(?:[-–:]\s*(\d+|end|last)?)?$", c, re.I)
            if not m:
                errors.append("Cannot read '%s'" % c)
                continue
            a = int(m.group(1))
            b_raw = m.group(2)
            if b_raw is None:
                b = a if "-" not in c and ":" not in c else self.n
            elif str(b_raw).lower() in ("end", "last"):
                b = self.n
            else:
                b = int(b_raw)
            if a < 1 or b > self.n or a > b:
                errors.append("Range '%s' is outside 1-%d" % (c, self.n))
                continue
            ranges.append((a - 1, b))
        return ranges, errors

    def coverage_warnings(self, ranges):
        """Missing / duplicated pages across the chosen ranges."""
        seen, dup = set(), set()
        for s, e in ranges:
            for p in range(s, e):
                if p in seen:
                    dup.add(p + 1)
                seen.add(p)
        missing = [p + 1 for p in range(self.n) if p not in seen]
        return sorted(missing), sorted(dup)

    # ---------- output ----------
    def write_parts(self, ranges, outdir, prefix=None):
        os.makedirs(outdir, exist_ok=True)
        prefix = prefix or self.stem
        paths = []
        for i, (s, e) in enumerate(ranges, start=1):
            path = os.path.join(
                outdir, "%s_part%02d_p%d-%d.pdf" % (prefix, i, s + 1, e)
            )
            with open(path, "wb") as fh:
                fh.write(self._write_bytes(s, e))
            paths.append(path)
        return paths

    @staticmethod
    def zip_files(paths, zip_path):
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for p in paths:
                zf.write(p, os.path.basename(p))
        return zip_path


def rows_for_display(parts):
    """Table rows for a parts list."""
    rows = []
    for p in parts:
        rows.append(
            {
                "Part": p["part"],
                "Pages": "%d-%d" % (p["start"] + 1, p["end"]),
                "No. of pages": p["end"] - p["start"],
                "Actual size": human(p["size"]),
                "Target": human(p["target"]) if p.get("target") else "-",
                "Within target": ("Yes" if p["ok"] else "NO - single page too big")
                if p.get("target")
                else "-",
            }
        )
    return rows


# Shared state between cells
STATE = {"engine": None, "ranges_text": "", "ranges": [], "files": []}

print("Setup done. Colab:", IN_COLAB, "| pandas:", HAS_PANDAS)
print("Next: run Cell 2 and upload your PDF.")

In [ ]:
# =====================================================================
# CELL 2 - UPLOAD THE PDF AND MEASURE IT
# Shows: total file size, page count, and the real size of every page.
# =====================================================================
data, fname = None, None

if IN_COLAB:
    print("Choose ONE pdf file...")
    up = colab_files.upload()
    pdfs = [k for k in up if k.lower().endswith(".pdf")]
    if not pdfs:
        raise ValueError("No .pdf file was uploaded.")
    fname = pdfs[0]
    data = up[fname]
    if len(pdfs) > 1:
        print("More than one PDF uploaded - using the first one:", fname)
else:
    # Outside Colab: put the path of a local file here
    fname = "input.pdf"
    data = open(fname, "rb").read()

eng = SplitEngine(data, fname)
if eng.encrypted:
    print("WARNING: this PDF is encrypted/password protected. Splitting may fail.")

print("\nFile          :", fname)
print("File size     :", human(eng.file_size))
print("Total pages   :", eng.n)
print("\nMeasuring each page (writing it out to get its true size)...")

def _prog(i, n):
    step = max(1, n // 10)
    if i % step == 0 or i == n:
        print("   %d / %d pages measured" % (i, n))

eng.analyse(progress=_prog)
STATE["engine"] = eng
STATE["ranges_text"] = "1-%d" % eng.n

avg = sum(eng.page_sizes) / float(eng.n)
big = max(range(eng.n), key=lambda i: eng.page_sizes[i])
small = min(range(eng.n), key=lambda i: eng.page_sizes[i])

print("\n----- SUMMARY -----")
print("Size if re-saved as one file :", human(eng.rewrite_size))
print("Average page                 :", human(avg))
print("Heaviest page                : page %d (%s)" % (big + 1, human(eng.page_sizes[big])))
print("Lightest page                : page %d (%s)" % (small + 1, human(eng.page_sizes[small])))
print("\nRough guide - how many parts for a given target size:")
for t in [1, 2, 5, 10, 20]:
    k = max(1, int(-(-eng.rewrite_size // (t * MB))))
    print("   target %2d MB  ->  about %d part(s)" % (t, k))

if HAS_PANDAS:
    df_pages = pd.DataFrame(
        {"Page": range(1, eng.n + 1),
         "Size (bytes)": eng.page_sizes,
         "Size": [human(s) for s in eng.page_sizes]}
    )
    print("\nPage-wise sizes (first 25 rows):")
    display(df_pages.head(25))
    STATE["page_table"] = df_pages

In [ ]:
# =====================================================================
# CELL 3 - RECOMMENDED SPLIT (choose your desired size)
#   Mode 1: same target size for every part
#   Mode 2: a different size for each part, e.g. "5, 10, 2, 5"
#           (if pages are left over, the last size keeps repeating)
#   Mode 3: fixed number of parts with equal page counts
# The recommended ranges are copied into Cell 4, where you can change them.
# =====================================================================
eng = STATE["engine"]
assert eng is not None, "Run Cell 2 first."

mode = widgets.RadioButtons(
    options=[("Same target size for every part", "same"),
             ("Different size per part (comma list, MB)", "list"),
             ("Fixed number of parts (equal page count)", "count")],
    value="same", description="Mode:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"))

size_mb = widgets.FloatText(value=5.0, description="Target size (MB):",
                            style={"description_width": "initial"})
size_list = widgets.Text(value="5, 10, 5", description="Sizes per part (MB):",
                         style={"description_width": "initial"},
                         layout=widgets.Layout(width="420px"))
n_parts = widgets.IntText(value=3, description="Number of parts:",
                          style={"description_width": "initial"})
btn = widgets.Button(description="Recommend split", button_style="primary")
out = widgets.Output()

def _show(parts):
    rows = rows_for_display(parts)
    if HAS_PANDAS:
        display(pd.DataFrame(rows))
    else:
        for r in rows:
            print(r)
    txt = SplitEngine.parts_to_text(parts)
    STATE["ranges_text"] = txt
    STATE["ranges"] = [(p["start"], p["end"]) for p in parts]
    total = sum(p["size"] for p in parts)
    print("\nParts: %d | Pages covered: 1-%d | Sum of parts: %s"
          % (len(parts), eng.n, human(total)))
    print("Page ranges  ->  " + txt)
    print("\nThese ranges are now loaded into Cell 4. Run Cell 4 to edit them,")
    print("or go straight to Cell 5 to split.")
    over = [p["part"] for p in parts if not p["ok"]]
    if over:
        print("\nNOTE: part(s) %s are over the target because a single page "
              "is already bigger than the size you asked for." % over)

def on_click(b):
    with out:
        out.clear_output()
        if mode.value == "same":
            if size_mb.value <= 0:
                print("Enter a target size greater than 0.")
                return
            parts = eng.plan_by_size([int(size_mb.value * MB)])
        elif mode.value == "list":
            try:
                vals = [float(x) for x in size_list.value.replace(";", ",").split(",") if x.strip()]
            except ValueError:
                print("Could not read the size list. Example: 5, 10, 2.5")
                return
            if not vals or min(vals) <= 0:
                print("All sizes must be greater than 0.")
                return
            parts = eng.plan_by_size([int(v * MB) for v in vals])
        else:
            parts = eng.plan_by_count(n_parts.value)
        _show(parts)

btn.on_click(on_click)
display(widgets.VBox([mode, size_mb, size_list, n_parts, btn, out]))

In [ ]:
# =====================================================================
# CELL 4 - MANUAL PAGE SELECTION (this is the one that decides the split)
#   Type any ranges you like:  1-12, 13-30, 31-end
#   - parts may be of different sizes / page counts
#   - a single page is just the number, e.g.  7
#   - pages may be skipped or repeated; you only get a warning
#   Press "Check sizes" to see the real size of each part before splitting.
# =====================================================================
eng = STATE["engine"]
assert eng is not None, "Run Cell 2 first."

box = widgets.Textarea(
    value=STATE.get("ranges_text", "1-%d" % eng.n),
    description="Page ranges:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="760px", height="90px"))
load_btn = widgets.Button(description="Load recommended (from Cell 3)")
check_btn = widgets.Button(description="Check sizes", button_style="primary")
out4 = widgets.Output()

def on_load(b):
    box.value = STATE.get("ranges_text", box.value)

def on_check(b):
    with out4:
        out4.clear_output()
        ranges, errors = eng.parse_ranges(box.value)
        for e in errors:
            print("ERROR:", e)
        if not ranges:
            print("No valid ranges found. Example: 1-12, 13-30, 31-end")
            return
        parts = eng.size_parts(ranges)
        rows = rows_for_display(parts)
        if HAS_PANDAS:
            display(pd.DataFrame(rows)[["Part", "Pages", "No. of pages", "Actual size"]])
        else:
            for r in rows:
                print(r)
        missing, dup = eng.coverage_warnings(ranges)
        print("\nParts: %d | Sum of parts: %s | Original: %s"
              % (len(parts), human(sum(p["size"] for p in parts)), human(eng.file_size)))
        if missing:
            print("WARNING: these pages are NOT in any part -> %s"
                  % (", ".join(map(str, missing))[:300]))
        if dup:
            print("WARNING: these pages appear in more than one part -> %s"
                  % (", ".join(map(str, dup))[:300]))
        if not missing and not dup:
            print("All %d pages are covered exactly once." % eng.n)
        STATE["ranges"] = ranges
        STATE["ranges_text"] = box.value
        print("\nSelection saved. Run Cell 5 to split and download.")

load_btn.on_click(on_load)
check_btn.on_click(on_check)
display(widgets.VBox([box, widgets.HBox([load_btn, check_btn]), out4]))

In [ ]:
# =====================================================================
# CELL 5 - SPLIT AND DOWNLOAD
# Uses the ranges last confirmed in Cell 4 (or recommended in Cell 3).
# =====================================================================
import os, datetime

eng = STATE["engine"]
assert eng is not None, "Run Cell 2 first."

prefix_box = widgets.Text(value=eng.stem, description="File name prefix:",
                          style={"description_width": "initial"},
                          layout=widgets.Layout(width="420px"))
zip_chk = widgets.Checkbox(value=True, description="Download as one .zip")
sep_chk = widgets.Checkbox(value=False, description="Also download each PDF separately")
split_btn = widgets.Button(description="Split & Download", button_style="success")
out5 = widgets.Output()

OUTDIR = "/content/pdf_splits" if IN_COLAB else "pdf_splits"

def on_split(b):
    with out5:
        out5.clear_output()
        ranges = STATE.get("ranges")
        if not ranges:
            ranges, errors = eng.parse_ranges(STATE.get("ranges_text", ""))
            for e in errors:
                print("ERROR:", e)
        if not ranges:
            print("No page ranges selected. Use Cell 3 or Cell 4 first.")
            return
        if os.path.isdir(OUTDIR):
            for f in os.listdir(OUTDIR):
                os.remove(os.path.join(OUTDIR, f))
        paths = eng.write_parts(ranges, OUTDIR, prefix=prefix_box.value.strip() or eng.stem)
        print("Created %d file(s) in %s :\n" % (len(paths), OUTDIR))
        for p in paths:
            print("   %-55s %s" % (os.path.basename(p), human(os.path.getsize(p))))
        STATE["files"] = paths

        if IN_COLAB:
            if zip_chk.value:
                stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
                zpath = os.path.join(OUTDIR, "%s_split_%s.zip" % (
                    (prefix_box.value.strip() or eng.stem), stamp))
                SplitEngine.zip_files(paths, zpath)
                print("\nZip: %s (%s)" % (os.path.basename(zpath), human(os.path.getsize(zpath))))
                colab_files.download(zpath)
            if sep_chk.value:
                for p in paths:
                    colab_files.download(p)
            if not zip_chk.value and not sep_chk.value:
                print("\nNothing downloaded - tick one of the boxes above.")
        else:
            print("\nNot running in Colab - files are saved in the folder above.")

split_btn.on_click(on_split)
display(widgets.VBox([prefix_box, zip_chk, sep_chk, split_btn, out5]))